# 🛡️ Glu-Stock: 00a_MODEL_RETRAINING_RF
**Phase**: Dynamic Cross-Sectional Intelligence (RF Brain)

This notebook trains the Random Forest 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the model stays relevant to the most liquid assets.

In [11]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib lxml html5lib python-dotenv



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Web Fetchers)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, firestore
from datetime import datetime

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as url:
            data = json.loads(url.read().decode())
            return [f"{t}.JK" for t in data]
    except Exception as e:
        print(f"⚠️ Github LQ45 fetch failed: {e}. Using highly-curated fallback LQ45 list.")
    return fallback

def get_full_idx_universe():
    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")
    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    
    # 1. Try Official IDX API
    try:
        import urllib.request
        hdrs = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://www.idx.co.id/'
        }
        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers=hdrs)
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            if 'data' in data:
                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]
                if tickers:
                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")
                    return list(set(tickers)) 
    except Exception as e:
        print(f"⚠️ Official IDX API failed: {e}. Trying Github Proxy...")
        
    # 2. Try Github Alternative
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]
            if tickers:
                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")
                return list(set(tickers))
    except Exception as e:
        print(f"⚠️ Full fetch failed: {e}. Falling back to MASTER 259 Papan Utama list.")
        
    return fallback
\n

SyntaxError: unexpected character after line continuation character (1535006911.py, line 2)

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Panel Training Pipeline)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np
import ta

def build_panel_data(universe, period="5y"):
    all_X, all_y = [], []
    print(f"📉 Fetching {period} of data for {len(universe)} tickers...")
    feature_names = ['Returns', 'RSI', 'MACD', 'BB_High', 'BB_Low']
    
    for ticker in universe:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > 100:
            # Squeeze to 1D Series to prevent Dimension Errors from yfinance tuple columns
            close_series = df['Close'].squeeze()
            
            # 1. Price Action
            df['Returns'] = close_series.pct_change()
            
            # 2. Momentum (RSI)
            df['RSI'] = ta.momentum.RSIIndicator(close=close_series, window=14).rsi()
            
            # 3. Trend (MACD)
            macd = ta.trend.MACD(close=close_series)
            df['MACD'] = macd.macd_diff()
            
            # 4. Volatility (Bollinger Bands)
            bollinger = ta.volatility.BollingerBands(close=close_series, window=20, window_dev=2)
            df['BB_High'] = bollinger.bollinger_hband_indicator()
            df['BB_Low'] = bollinger.bollinger_lband_indicator()
            
            df = df.dropna()
            
            # Use iloc internally for strict array selection
            X = df[feature_names].values
            y = (df['Close'].shift(-1) > df['Close']).iloc[:-1].values.astype(int)
            all_X.append(X[:len(y)])
            all_y.append(y)
            
    print(f"✅ Successfully aggregated market data.")
    if len(all_X) == 0:
        return None, None, feature_names
    return np.vstack(all_X), np.concatenate(all_y), feature_names

def train_rf(X_train, y_train):
    print(f"🧠 Training Institutional RF Brain on {len(y_train)} samples...")
    model = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    return model


In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = "/kaggle/working/"
    
    universe = get_full_idx_universe()
    
    # 1. Build Panel & Train RF
    X, y, feature_names = build_panel_data(universe)
    if X is None:
        print('⚠️ Retraining aborted due to data fetch failure.')
        return
        
    # 🛑 CRITICAL FIX: shuffle=False to prevent Time-Leakage!
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)
    
    rf_model = train_rf(X_train, y_train)
    train_acc = rf_model.score(X_train, y_train)
    val_acc = rf_model.score(X_test, y_test)
    
    brain_data = {
        "model": rf_model, 
        "features": feature_names, 
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "trained_at": datetime.now().isoformat()
    }
    joblib.dump(brain_data, os.path.join(output_dir, "glu_brain_v1.joblib"))
    
    rich_log = f"RF Training Complete.\n🎓 Training Acc: {train_acc:.2%}\n🛡️ OOS Validation Acc: {val_acc:.2%}\n📊 Target Universe: {len(universe)} symbols\n🧠 Features: {len(feature_names)}"
    fb.log_event("RETRAINING_RF", rich_log)
    print(f"✅ RF Model updated successfully: Train Acc {train_acc:.3f} | OOS Val Acc {val_acc:.3f}")

run_retrain()


🌐 Fetching latest LQ45 constituents...
📉 Fetching 5y of data for 45 tickers...



1 Failed download:
['ACES.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ADRO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AKRA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMMN.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMRT.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ANTM.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ARTO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ASII.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBCA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBNI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBRI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBTN.JK']: TypeError("'NoneType' object is 

✅ Successfully aggregated market data.


ValueError: need at least one array to concatenate